# QuestMasters — DM Multiagente (MAS) en Colab

Corre Gemma 4 26B-A4B-it (4-bit) **sin LoRAs** — cada agente usa su propio system prompt.
**Todo el pipeline usa un único modelo (Gemma 4)** — router, agentes, narrador y extracción.

**Pipeline:** `router (Gemma 4) → L1 (ChromaDB) → reglas (L1+L4) → npc (L1+L2+L3) → mundo (L1+L2+L3) → narrador (L1+L2) → post-turno (W→L1, W→L2, W→L3)`

**Capas de memoria:**
- **L1 Working** — ChromaDB por sesión (contexto reciente, búsqueda semántica sin LLM)
- **L2 Episódica** — SQLite + ChromaDB por sesión (turnos pasados, búsqueda semántica)
- **L3 Semántica** — Grafo NetworkX por sesión (PNJs, lugares, relaciones entre entidades)
- **L4 Procedural** — ChromaDB compartido (reglas SRD de D&D 5e, estático)

**Pasos:**
1. Configura los secretos en el panel izquierdo (icono de llave):
   - `HF_TOKEN` — tu token de HuggingFace
   - `NGROK_TOKEN` — tu token de ngrok
2. Ejecuta todas las celdas
3. Copia la URL de ngrok
4. En tu `.dev.vars` del backend:
   ```
   DM_MODEL_ENDPOINT_MAS=<url_ngrok>
   DM_USE_RUNPOD=false
   ```

In [ ]:
!pip install -q transformers accelerate bitsandbytes safetensors \
    huggingface_hub fastapi uvicorn pyngrok nest_asyncio \
    networkx chromadb pydantic numpy

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
NGROK_TOKEN = userdata.get("NGROK_TOKEN")

print(f"HF_TOKEN: {'OK' if os.environ['HF_TOKEN'] else 'FALTA'}")
print(f"NGROK: {'OK' if NGROK_TOKEN else 'FALTA'}")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

## 1. Cargar modelo base (sin LoRA)

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

HF_TOKEN = os.environ["HF_TOKEN"]
BASE_MODEL_ID = "google/gemma-4-26B-A4B-it"

print(f"Cargando {BASE_MODEL_ID} (4-bit NF4, sin LoRA)...")
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)

print("Modelo base listo (sin LoRA — puro system prompt).")
free, total = torch.cuda.mem_get_info(0)
print(f"VRAM usada: {(total - free) / 1024**3:.1f} GB / {total / 1024**3:.1f} GB")

## 2. Memoria de 4 capas (L1 + L2 + L3 + L4)

In [ ]:
import asyncio
import json
import re
import sqlite3
import time
import uuid
import logging
from pathlib import Path
from threading import Thread
from typing import Any, Generator

import chromadb
import networkx as nx
import torch
from transformers import TextIteratorStreamer

import nest_asyncio
nest_asyncio.apply()

log = logging.getLogger("dm-mas")

SESSIONS_DIR = Path("/content/sessions")
SESSIONS_DIR.mkdir(parents=True, exist_ok=True)
SRD_CHROMA_PATH = "/content/shared/l4_srd_chroma"
Path(SRD_CHROMA_PATH).mkdir(parents=True, exist_ok=True)


def _session_path(session_id: str) -> Path:
    path = SESSIONS_DIR / session_id
    path.mkdir(parents=True, exist_ok=True)
    return path


# ═══════════════════════════════════════════════════════════════════════
# L1 — Memoria de Trabajo (ChromaDB por sesión, sin LLM)
# ═══════════════════════════════════════════════════════════════════════

def _l1_collection(session_id: str) -> chromadb.Collection:
    chroma_path = str(_session_path(session_id) / "l1_chroma")
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_or_create_collection("working_memory")


def l1_retrieve(session_id: str, query: str) -> str:
    collection = _l1_collection(session_id)
    if collection.count() == 0:
        return ""
    n = min(collection.count(), 3)
    results = collection.query(query_texts=[query], n_results=n)
    if not results["documents"] or not results["documents"][0]:
        return ""
    return "\n\n".join(results["documents"][0])


def l1_insert_turn(
    session_id: str, turn: int,
    player_input: str, dm_response: str,
    arbiter_ruling: str = "", npc_responses: str = "", world_state: str = "",
) -> None:
    parts = [f"--- Turno {turn} ---"]
    if player_input:
        parts.append(f"Acción del jugador: {player_input}")
    if arbiter_ruling:
        parts.append(f"Resolución de reglas: {arbiter_ruling}")
    if npc_responses:
        parts.append(f"Reacciones PNJs: {npc_responses}")
    if world_state:
        parts.append(f"Estado del mundo: {world_state}")
    parts.append(f"Narración del DM: {dm_response}")

    collection = _l1_collection(session_id)
    collection.upsert(
        ids=[f"turn-{turn}"],
        documents=["\n".join(parts)],
        metadatas=[{"turn": turn}],
    )
    print(f"[L1] Turno {turn} insertado en memoria de trabajo")


# ═══════════════════════════════════════════════════════════════════════
# L2 — Memoria Episódica (SQLite + ChromaDB por sesión)
# ═══════════════════════════════════════════════════════════════════════

def _init_l2_db(session_id: str) -> sqlite3.Connection:
    db_path = _session_path(session_id) / "l2.db"
    conn = sqlite3.connect(str(db_path))
    conn.execute("""
        CREATE TABLE IF NOT EXISTS episodes (
            id      TEXT PRIMARY KEY,
            turn    INTEGER NOT NULL,
            role    TEXT NOT NULL,
            content TEXT NOT NULL,
            metadata TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    return conn


def _l2_collection(session_id: str) -> chromadb.Collection:
    chroma_path = str(_session_path(session_id) / "l2_chroma")
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_or_create_collection("episodes")


def l2_save_episode(session_id: str, turn: int, role: str, content: str, metadata: dict) -> str:
    episode_id = str(uuid.uuid4())
    conn = _init_l2_db(session_id)
    conn.execute(
        "INSERT INTO episodes (id, turn, role, content, metadata) VALUES (?, ?, ?, ?, ?)",
        (episode_id, turn, role, content, json.dumps(metadata)),
    )
    conn.commit()
    conn.close()
    _l2_collection(session_id).add(
        ids=[episode_id], documents=[content],
        metadatas=[{"turn": turn, "role": role}],
    )
    return episode_id


def l2_retrieve(session_id: str, query: str, n_results: int = 5) -> list[dict]:
    collection = _l2_collection(session_id)
    if collection.count() == 0:
        return []
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {"id": results["ids"][0][i], "content": doc, "metadata": results["metadatas"][0][i]}
        for i, doc in enumerate(results["documents"][0])
    ]


# ═══════════════════════════════════════════════════════════════════════
# L3 — Memoria Semántica (Grafo NetworkX por sesión)
# ═══════════════════════════════════════════════════════════════════════

def _l3_graph_path(session_id: str) -> Path:
    return _session_path(session_id) / "l3_graph.json"


def l3_load_graph(session_id: str) -> nx.DiGraph:
    path = _l3_graph_path(session_id)
    if not path.exists():
        return nx.DiGraph()
    return nx.node_link_graph(json.loads(path.read_text()))


def l3_save_graph(session_id: str, graph: nx.DiGraph) -> None:
    _l3_graph_path(session_id).write_text(json.dumps(nx.node_link_data(graph)))


def l3_query_neighbors(graph: nx.DiGraph, entity_id: str, depth: int = 1) -> list[dict]:
    if entity_id not in graph:
        return []
    nodes = nx.single_source_shortest_path_length(graph, entity_id, cutoff=depth)
    results = []
    for node_id in nodes:
        data = dict(graph.nodes[node_id])
        data["id"] = node_id
        results.append(data)
    return results


def l3_update_from_extraction(session_id: str, entities: list[dict]) -> None:
    graph = l3_load_graph(session_id)
    for item in entities:
        entity_id = item["id"]
        entity_type = item.get("type", "unknown")
        attrs = {k: v for k, v in item.items() if k not in ("id", "type", "relations")}
        graph.add_node(entity_id, type=entity_type, **attrs)
        for rel in item.get("relations", []):
            graph.add_edge(entity_id, rel["target"], relation=rel["type"])
    l3_save_graph(session_id, graph)


# ═══════════════════════════════════════════════════════════════════════
# L4 — Memoria Procedural (ChromaDB compartido con reglas SRD)
# ═══════════════════════════════════════════════════════════════════════

def _l4_collection() -> chromadb.Collection:
    client = chromadb.PersistentClient(path=SRD_CHROMA_PATH)
    return client.get_or_create_collection("srd_rules")


def l4_query_rules(query: str, n_results: int = 5) -> list[dict]:
    collection = _l4_collection()
    if collection.count() == 0:
        return []
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {"id": results["ids"][0][i], "content": doc, "metadata": results["metadatas"][0][i]}
        for i, doc in enumerate(results["documents"][0])
    ]


print("[memoria] L1 (working/ChromaDB) + L2 (episódica) + L3 (semántica) + L4 (SRD procedural)")
print("L1 usa ChromaDB directo — sin llamadas LLM para inserción.")

## 3. Agentes multiagente (system prompts)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Generación compartida — todos los agentes usan esto
# ═══════════════════════════════════════════════════════════════════════

class TokenUsage:
    def __init__(self):
        self.input_tokens = 0
        self.output_tokens = 0

    def add(self, input_t: int, output_t: int):
        self.input_tokens += input_t
        self.output_tokens += output_t


def generate_agent_response(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    usage: TokenUsage | None = None,
) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=False,
    ).to(model.device)
    n_input = input_ids.shape[1]

    use_sampling = temperature > 1e-5
    gen_kwargs: dict = {
        "input_ids": input_ids,
        "max_new_tokens": max_new_tokens,
        "do_sample": use_sampling,
        "repetition_penalty": 1.3,
    }
    if use_sampling:
        gen_kwargs.update(temperature=temperature, top_p=0.9, top_k=40)

    output_ids = model.generate(**gen_kwargs)
    n_output = output_ids.shape[1] - n_input
    if usage:
        usage.add(n_input, n_output)
    return tokenizer.decode(output_ids[0][n_input:], skip_special_tokens=True).strip()


def stream_agent_response(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    usage: TokenUsage | None = None,
) -> Generator[str, None, str]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=False,
    ).to(model.device)
    n_input = input_ids.shape[1]
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    use_sampling = temperature > 1e-5
    gen_kwargs: dict = {
        "input_ids": input_ids,
        "max_new_tokens": max_new_tokens,
        "do_sample": use_sampling,
        "repetition_penalty": 1.3,
        "streamer": streamer,
    }
    if use_sampling:
        gen_kwargs.update(temperature=temperature, top_p=0.9, top_k=40)

    Thread(target=model.generate, kwargs=gen_kwargs, daemon=True).start()
    full = ""
    n_output = 0
    for token in streamer:
        full += token
        n_output += 1
        yield token
    if usage:
        usage.add(n_input, n_output)
    return full


# ═══════════════════════════════════════════════════════════════════════
# Router — clasifica intención usando Gemma 4 (fallback: keywords)
# ═══════════════════════════════════════════════════════════════════════

_ROUTER_SYSTEM = (
    "Eres un clasificador de intenciones para una mesa de D&D 5e. "
    "Dado el input del jugador, determina qué agentes necesitan participar.\n\n"
    "Agentes:\n"
    "- memory: recuperar recuerdos de turnos anteriores. "
    "Necesario cuando se referencia el pasado o hay suficiente historial.\n"
    "- arbiter: consultar reglas SRD. Necesario para combate, magia, tiradas, habilidades.\n"
    "- npc: reacciones de PNJs. Necesario cuando el jugador interactúa con alguien.\n"
    "- world: cambios en el mundo. Necesario cuando la acción altera el entorno.\n\n"
    'Responde SOLO con JSON válido, sin texto adicional:\n'
    '{"needs_memory": true/false, "needs_arbiter": true/false, "needs_npc": true/false, "needs_world": true/false}'
)

_ARBITER_KW = {"ataco","golpeo","lanzo","hechizo","conjuro","intento","escalo","salto",
    "fuerzo","empujo","escondo","examino","investigo","busco","tirada","dado",
    "teléfono","celular","pistola","computadora"}
_NPC_KW = {"hablo","digo","pregunto","convenzo","persuado","intimido","engaño",
    "negocio","amenazo","saludo","pido","le digo","le pregunto","respondo"}
_WORLD_KW = {"destruyo","incendio","quemo","rompo","mato","robo","saqueo",
    "viajo","camino","corro","huyo","escapo","salgo","entro","exploro",
    "espero","descanso","duermo"}
_MEMORY_KW = {"recuerdo","antes","previamente","mencionó","dijo","hizo","anterior"}


def _kw_match(text: str, kws: set[str]) -> bool:
    t = re.sub(r"[^\w\sáéíóúñü]", "", text.lower())
    return any(k in t for k in kws)


def _parse_route_json(raw: str) -> dict | None:
    start = raw.find("{")
    end = raw.rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        parsed = json.loads(raw[start:end])
        return {
            "needs_memory": bool(parsed.get("needs_memory", False)),
            "needs_arbiter": bool(parsed.get("needs_arbiter", False)),
            "needs_npc": bool(parsed.get("needs_npc", False)),
            "needs_world": bool(parsed.get("needs_world", False)),
        }
    except (json.JSONDecodeError, AttributeError):
        return None


def classify_intent(player_input: str, campaign: str, turn_count: int, usage: TokenUsage) -> dict:
    try:
        raw = generate_agent_response(
            _ROUTER_SYSTEM,
            f"Campaña: {campaign}\nTurno: {turn_count}\nJugador: {player_input}",
            max_new_tokens=64,
            temperature=0.0,
            usage=usage,
        )
        result = _parse_route_json(raw)
        if result:
            if turn_count >= 3:
                result["needs_memory"] = True
            return result
    except Exception as exc:
        print(f"[router] Gemma falló ({exc}), usando keywords")

    return {
        "needs_memory": turn_count >= 3 or _kw_match(player_input, _MEMORY_KW),
        "needs_arbiter": _kw_match(player_input, _ARBITER_KW),
        "needs_npc": _kw_match(player_input, _NPC_KW),
        "needs_world": _kw_match(player_input, _WORLD_KW),
    }


# ═══════════════════════════════════════════════════════════════════════
# Agente de Reglas (árbitro) — lee L1 del bb, consulta L4 directo
# ═══════════════════════════════════════════════════════════════════════

ARBITER_SYSTEM = (
    "Eres el Agente de Reglas de una mesa de D&D 5e. Tu rol es ser el consultor "
    "directo del SRD y garantizar la coherencia lógica del mundo.\n\n"
    "RESPONSABILIDADES:\n"
    "1. Evaluar si la acción del jugador es válida según el SRD 5e.\n"
    "2. Determinar si requiere tirada, qué habilidad y CD (entre 8 y 20).\n"
    "3. Verificar coherencia lógica: sin anacronismos salvo justificación de campaña.\n"
    "4. Identificar restricciones de clase, raza o nivel.\n"
    "5. Señalar consecuencias mecánicas (daño, condiciones).\n\n"
    "FORMATO (conciso, en español):\n"
    "- VALIDEZ: [válida / inválida / parcial] — razón\n"
    "- TIRADA: [no requerida / Habilidad (CD N)] — justificación\n"
    "- COHERENCIA: [coherente / incoherente] — qué rompe la lógica\n"
    "- MECÁNICA: efectos relevantes\n"
    "- NOTA: regla SRD que el narrador deba considerar"
)


def agent_arbiter(
    campaign: str, player_input: str, history: list[dict],
    l1_context: str, usage: TokenUsage,
) -> str:
    recent = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history[-4:])
    l4_context = l4_query_rules(player_input, n_results=5)
    rules = "\n".join(r["content"] for r in l4_context[:5])

    parts = [f"CAMPAÑA: {campaign}"]
    if l1_context:
        parts.append(f"CONTEXTO RECIENTE (L1):\n{l1_context}")
    if rules:
        parts.append(f"REGLAS SRD RELEVANTES (L4):\n{rules}")
    if recent:
        parts.append(f"HISTORIAL:\n{recent}")
    parts.append(f"ACCIÓN: {player_input}")
    return generate_agent_response(ARBITER_SYSTEM, "\n\n".join(parts), max_new_tokens=256, temperature=0.3, usage=usage)


# ═══════════════════════════════════════════════════════════════════════
# Agente de NPCs — lee L1 del bb, consulta L2+L3 directo
# ═══════════════════════════════════════════════════════════════════════

NPC_SYSTEM = (
    "Eres el Agente de NPCs de una mesa de D&D 5e. Tu rol es crear arquetipos de "
    "PNJs y definir cómo se comportan en cada situación.\n\n"
    "RESPONSABILIDADES:\n"
    "1. Definir personalidad, motivaciones y objetivos de cada PNJ presente.\n"
    "2. Determinar cómo reacciona cada PNJ ante la acción del jugador.\n"
    "3. Generar diálogo coherente con la personalidad del PNJ.\n"
    "4. Mantener consistencia: un cobarde no se vuelve valiente sin razón.\n"
    "5. Si no hay PNJs presentes, indicar si alguien cercano reaccionaría.\n\n"
    "REGLAS:\n"
    "- Los PNJs tienen voluntad propia, no obedecen automáticamente.\n"
    "- Reacciones proporcionales a la situación.\n"
    "- Diálogo natural, frases cortas, con personalidad.\n\n"
    "FORMATO por PNJ:\n"
    "- NOMBRE / ACTITUD / REACCIÓN / DIÁLOGO (entre comillas) / INTENCIÓN"
)


def agent_npc(
    session_id: str, campaign: str, player_input: str, arbiter_ruling: str,
    history: list[dict], l1_context: str, usage: TokenUsage,
) -> str:
    l2_context = l2_retrieve(session_id, player_input, n_results=5)
    graph = l3_load_graph(session_id)
    last_message = (history or [{}])[-1].get("content", "")
    relevant_ids = [n for n in graph.nodes() if n.lower() in last_message.lower()][:3]
    l3_context = []
    for eid in relevant_ids:
        l3_context.extend(l3_query_neighbors(graph, eid, depth=1))

    recent = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history[-4:])
    l2_text = "\n".join(ep["content"] for ep in l2_context[:3])
    npcs = [e for e in l3_context if e.get("type") == "npc"]
    npcs_text = "\n".join(f"- {e.get('name', '?')}: {e.get('personality', 'sin definir')}" for e in npcs)

    parts = [f"CAMPAÑA: {campaign}"]
    if l1_context:
        parts.append(f"CONTEXTO RECIENTE (L1):\n{l1_context}")
    if npcs_text:
        parts.append(f"PNJs CONOCIDOS (L3):\n{npcs_text}")
    if l2_text:
        parts.append(f"CONTEXTO EPISÓDICO (L2):\n{l2_text}")
    if arbiter_ruling:
        parts.append(f"RESOLUCIÓN ÁRBITRO:\n{arbiter_ruling}")
    if recent:
        parts.append(f"HISTORIAL:\n{recent}")
    parts.append(f"ACCIÓN: {player_input}")
    return generate_agent_response(NPC_SYSTEM, "\n\n".join(parts), max_new_tokens=300, temperature=0.8, usage=usage)


# ═══════════════════════════════════════════════════════════════════════
# Agente del Mundo — lee L1 del bb, consulta L2+L3 directo
# ═══════════════════════════════════════════════════════════════════════

WORLD_SYSTEM = (
    "Eres el Agente del Mundo de una mesa de D&D 5e. Tu rol es registrar cambios "
    "importantes en el mundo y determinar cómo repercuten en PNJs, lugares y facciones.\n\n"
    "RESPONSABILIDADES:\n"
    "1. Identificar si la acción provoca un cambio significativo en el mundo.\n"
    "2. Determinar repercusiones: qué PNJs se afectan, cómo cambian sus actitudes.\n"
    "3. Registrar consecuencias a futuro (rumores, patrullas, venganzas).\n"
    "4. Mantener causalidad: acciones tienen efectos dominó lógicos.\n\n"
    "FORMATO (conciso, en español):\n"
    "- EVENTO: qué cambió (o 'Sin cambios significativos')\n"
    "- AFECTADOS: PNJs, lugares o facciones impactados\n"
    "- REPERCUSIÓN INMEDIATA: efecto ahora mismo\n"
    "- REPERCUSIÓN FUTURA: qué podría pasar después\n"
    "- AMBIENTE: cómo cambia el tono de la escena"
)


def agent_world(
    session_id: str, campaign: str, player_input: str, arbiter_ruling: str,
    npc_response: str, history: list[dict], l1_context: str, usage: TokenUsage,
) -> str:
    l2_context = l2_retrieve(session_id, player_input, n_results=5)
    graph = l3_load_graph(session_id)
    last_message = (history or [{}])[-1].get("content", "")
    relevant_ids = [n for n in graph.nodes() if n.lower() in last_message.lower()][:3]
    l3_context = []
    for eid in relevant_ids:
        l3_context.extend(l3_query_neighbors(graph, eid, depth=1))

    recent = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history[-4:])
    l2_text = "\n".join(ep["content"] for ep in l2_context[:3])
    entities_text = "\n".join(f"- [{e.get('type','?')}] {e.get('name','?')}" for e in l3_context)

    parts = [f"CAMPAÑA: {campaign}"]
    if l1_context:
        parts.append(f"CONTEXTO RECIENTE (L1):\n{l1_context}")
    if l2_text:
        parts.append(f"EVENTOS PREVIOS (L2):\n{l2_text}")
    if entities_text:
        parts.append(f"ENTIDADES CONOCIDAS (L3):\n{entities_text}")
    if arbiter_ruling:
        parts.append(f"RESOLUCIÓN ÁRBITRO:\n{arbiter_ruling}")
    if npc_response:
        parts.append(f"REACCIONES PNJs:\n{npc_response}")
    if recent:
        parts.append(f"HISTORIAL:\n{recent}")
    parts.append(f"ACCIÓN: {player_input}")
    return generate_agent_response(WORLD_SYSTEM, "\n\n".join(parts), max_new_tokens=300, temperature=0.5, usage=usage)


# ═══════════════════════════════════════════════════════════════════════
# Agente Narrativo (streaming) — lee L1 del bb, consulta L2 directo
# ═══════════════════════════════════════════════════════════════════════

NARRATOR_SYSTEM = (
    "Eres el Agente Narrativo de una mesa de D&D 5e. Sintetizas las decisiones de "
    "los demás agentes (reglas, NPCs, mundo) en una narración envolvente.\n\n"
    "REGLAS:\n"
    "1. El jugador NO debe notar que hay múltiples agentes. Suena como un solo DM.\n"
    "2. Incluye diálogo de PNJs de forma natural dentro de la narración.\n"
    "3. Si el árbitro pidió tirada, inclúyela con este formato: "
    "'Haz una tirada de [Habilidad] (CD [N])'. "
    "Puedes ofrecer opciones: 'Haz una tirada de Atletismo o Acrobacias (CD N)'.\n"
    "4. Si el mundo cambió, refléjalo en la atmósfera.\n\n"
    "PRIMER TURNO (cuando no hay acción del jugador):\n"
    "- Establece la escena: dónde están, qué ven, qué sienten, qué huelen.\n"
    "- Presenta el mundo y la situación con detalle inmersivo.\n"
    "- NO pidas tiradas en el primer turno. Termina con la situación planteada "
    "y deja que el jugador decida qué hacer.\n\n"
    "ESTILO:\n"
    "- Máximo 3 párrafos cortos. Oraciones de máximo 20 palabras.\n"
    "- Usa 'tú' para el jugador. Diálogos entre comillas con nombre del PNJ.\n"
    "- Alterna: A) Pedir tirada B) PNJ habla C) Algo inesperado D) Opciones.\n"
    "- NO termines todos los turnos con pregunta.\n\n"
    "TIRADAS: Después de pedir tirada, PARA. No narres el resultado.\n\n"
    "Responde SOLO con la narración. Sin encabezados, sin etiquetas, sin metadatos."
)


def build_narrator_prompt(
    session_id: str, campaign: str, player_input: str,
    arbiter_ruling: str, npc_response: str, world_state: str,
    history: list[dict], l1_context: str,
) -> str:
    l2_context = l2_retrieve(session_id, player_input, n_results=5)

    recent = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history[-6:])
    l2_text = "\n".join(ep["content"] for ep in l2_context[:3])

    parts = [f"CAMPAÑA: {campaign}"]
    if l1_context:
        parts.append(f"CONTEXTO RECIENTE (L1):\n{l1_context}")
    if l2_text:
        parts.append(f"CONTEXTO EPISÓDICO (L2):\n{l2_text}")
    if arbiter_ruling:
        parts.append(f"RESOLUCIÓN ÁRBITRO:\n{arbiter_ruling}")
    if npc_response:
        parts.append(f"REACCIONES PNJs:\n{npc_response}")
    if world_state:
        parts.append(f"ESTADO DEL MUNDO:\n{world_state}")
    if recent:
        parts.append(f"HISTORIAL:\n{recent}")
    if player_input:
        parts.append(f"ACCIÓN: {player_input}")
        parts.append("Narra la respuesta del DM integrando toda la información. Solo texto narrativo.")
    else:
        parts.append("PRIMER TURNO: No hay acción del jugador todavía. Establece la escena inicial de la campaña. Describe el entorno, la atmósfera y la situación. NO pidas tiradas.")
    return "\n\n".join(parts)


# ═══════════════════════════════════════════════════════════════════════
# Extracción post-turno — alimenta L2 y L3 después de cada turno
# Usa Gemma 4 con chat template para generar JSON estructurado.
# ═══════════════════════════════════════════════════════════════════════

_EXTRACTION_SYSTEM = (
    "Eres un extractor de información de partidas de D&D 5e.\n"
    "Dado un turno (acción del jugador + respuesta del DM), extrae eventos y entidades.\n"
    "Responde SOLO con JSON válido, sin texto adicional.\n\n"
    "Formato exacto:\n"
    '{"events": [{"description": "qué pasó", "participants": ["nombre"], "outcome": "resultado"}], '
    '"entities": [{"id": "nombre-en-kebab", "type": "npc|location|item|faction", "name": "Nombre", '
    '"relations": [{"target": "otro-id", "type": "tipo-relación"}]}]}\n\n'
    "Si no hay eventos o entidades nuevas, devuelve arrays vacíos."
)


def run_extraction(session_id: str, turn: int, dm_response: str, player_input: str) -> dict:
    user_prompt = f"Acción del jugador:\n{player_input}\n\nRespuesta del DM:\n{dm_response}"
    raw = generate_agent_response(
        _EXTRACTION_SYSTEM, user_prompt,
        max_new_tokens=512, temperature=0.0,
    )

    start, end = raw.find("{"), raw.rfind("}") + 1
    extracted = {"events": [], "entities": []}
    if start != -1 and end > 0:
        try:
            extracted = json.loads(raw[start:end])
        except json.JSONDecodeError:
            print(f"[extraction] JSON inválido: {raw[start:end][:200]}")

    for event in extracted.get("events", []):
        l2_save_episode(session_id, turn, "event", event.get("description", ""), event)

    if extracted.get("entities"):
        l3_update_from_extraction(session_id, extracted["entities"])

    return extracted


print("Agentes definidos: router, reglas, npc, mundo, narrador, extracción.")
print("TODO usa Gemma 4 — sin APIs externas para la orquestación del DM.")
print("L1 compartido vía blackboard | L2/L3/L4 consulta directa por agente.")

## 4. Servidor FastAPI + ngrok

In [ ]:
import asyncio
import gc
import queue
import traceback
from concurrent.futures import ThreadPoolExecutor
from typing import AsyncGenerator

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, ConfigDict, Field
from typing import Literal

app = FastAPI(title="QuestMasters DM MAS (Colab)")
_executor = ThreadPoolExecutor(max_workers=1)
_last_post_turn: Thread | None = None

_CAMEL_RE_1 = re.compile(r"(.)([A-Z][a-z]+)")
_CAMEL_RE_2 = re.compile(r"([a-z0-9])([A-Z])")


def _camel_to_snake(name: str) -> str:
    return _CAMEL_RE_2.sub(r"\1_\2", _CAMEL_RE_1.sub(r"\1_\2", name)).lower()


def _convert_keys(obj):
    if isinstance(obj, dict):
        return {_camel_to_snake(k): _convert_keys(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_convert_keys(i) for i in obj]
    return obj


class CharacterSnapshot(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    name: str
    race: str = ""
    class_name: str = Field(default="", alias="class")
    background: str = ""
    level: int = 1
    backstory: str = ""
    alignment: str = ""
    personality_traits: str = Field(default="", alias="personalityTraits")
    stats: dict[str, int] | None = None
    skill_proficiencies: list[str] = Field(default_factory=list, alias="skillProficiencies")
    expertise_skills: list[str] = Field(default_factory=list, alias="expertiseSkills")
    jack_of_all_trades: bool = Field(default=False, alias="jackOfAllTrades")
    reliable_talent: bool = Field(default=False, alias="reliableTalent")
    subclass: str = ""


class DmModelRequest(BaseModel):
    session_id: str
    architecture_type: Literal["monolithic", "mas"]
    model_id: str
    campaign_prompt: str
    characters: list[CharacterSnapshot]
    conversation_history: list[dict]
    player_input: str | None = None
    current_memory_snapshot: dict


def _cleanup_vram():
    gc.collect()
    torch.cuda.empty_cache()


def _run_mas_pipeline(request: DmModelRequest, q: queue.Queue) -> None:
    global _last_post_turn
    try:
        # Esperar post-turno anterior antes de usar la GPU
        if _last_post_turn is not None and _last_post_turn.is_alive():
            print("[pipeline] Esperando post-turno anterior...")
            _last_post_turn.join()

        _cleanup_vram()
        t_start = time.monotonic()
        player_input = request.player_input or ""
        campaign = request.campaign_prompt
        session_id = request.session_id
        history = request.conversation_history
        turn_count = len(history)
        usage = TokenUsage()

        # ── 1. Router: Gemma 4 → keywords fallback ──
        route = classify_intent(player_input, campaign, turn_count, usage)
        router_source = "Gemma 4"

        active = [k.replace("needs_", "") for k, v in route.items() if v]
        q.put({"type": "thinking", "agent": "router", "summary": f"[{router_source}] Consultando: {', '.join(active)}" if active else "Solo narración directa"})
        print(f"[router] ({router_source}) {route}")

        # ── 2. L1 Working Memory (una query compartida vía blackboard) ──
        l1_context = ""
        if route["needs_memory"]:
            q.put({"type": "thinking", "agent": "memoria", "summary": "Consultando memoria de trabajo (L1)..."})
            l1_context = l1_retrieve(session_id, player_input)

            if l1_context:
                preview = l1_context[:120].replace("\n", " ")
                q.put({"type": "thinking", "agent": "memoria", "summary": f"L1: {preview}..."})
            else:
                q.put({"type": "thinking", "agent": "memoria", "summary": "L1 vacío (primeros turnos), agentes usarán L2/L3/L4"})

        # ── 3. Árbitro de Reglas (bb:L1, consulta L4 directo) ──
        arbiter_ruling = ""
        if route["needs_arbiter"]:
            q.put({"type": "thinking", "agent": "reglas", "summary": "Evaluando reglas SRD y coherencia (L1+L4)..."})
            arbiter_ruling = agent_arbiter(campaign, player_input, history, l1_context, usage)
            q.put({"type": "thinking", "agent": "reglas", "summary": arbiter_ruling[:150]})
            print(f"[reglas] {arbiter_ruling[:100]}")

        # ── 4. NPCs (bb:L1, consulta L2+L3 directo) ──
        npc_response = ""
        if route["needs_npc"]:
            q.put({"type": "thinking", "agent": "npcs", "summary": "Determinando reacciones de PNJs (L1+L2+L3)..."})
            npc_response = agent_npc(session_id, campaign, player_input, arbiter_ruling, history, l1_context, usage)
            q.put({"type": "thinking", "agent": "npcs", "summary": npc_response[:150]})
            print(f"[npcs] {npc_response[:100]}")

        # ── 5. Mundo (bb:L1, consulta L2+L3 directo) ──
        world_state = ""
        if route["needs_world"]:
            q.put({"type": "thinking", "agent": "mundo", "summary": "Evaluando cambios en el mundo (L1+L2+L3)..."})
            world_state = agent_world(session_id, campaign, player_input, arbiter_ruling, npc_response, history, l1_context, usage)
            q.put({"type": "thinking", "agent": "mundo", "summary": world_state[:150]})
            print(f"[mundo] {world_state[:100]}")

        # ── 6. Narrador (streaming, bb:L1, consulta L2 directo) ──
        q.put({"type": "thinking", "agent": "narrador", "summary": "Narrando (L1+L2)..."})

        narrator_prompt = build_narrator_prompt(
            session_id, campaign, player_input,
            arbiter_ruling, npc_response, world_state,
            history, l1_context,
        )

        full_response = ""
        for token in stream_agent_response(NARRATOR_SYSTEM, narrator_prompt, max_new_tokens=512, temperature=0.8, usage=usage):
            full_response += token
            q.put({"type": "delta", "delta": token})

        latency_ms = int((time.monotonic() - t_start) * 1000)
        print(f"[tokens] input={usage.input_tokens} output={usage.output_tokens} total={usage.input_tokens + usage.output_tokens} | latency={latency_ms}ms")

        q.put({
            "type": "metadata",
            "metadata": {
                "memorySnapshot": {
                    "l1HasContext": bool(l1_context),
                    "turnCount": turn_count,
                    "routerSource": router_source,
                },
                "narrativeNotesDelta": [],
                "usage": {"inputTokens": usage.input_tokens, "outputTokens": usage.output_tokens},
                "latencyMs": latency_ms,
                "modelId": request.model_id,
                "agentsUsed": active,
            },
        })
        q.put({"type": "done"})
        q.put(None)

        # ── 7. Post-turno en BACKGROUND: W→L1 + W→L2 + W→L3 ──
        def _post_turn_work():
            _cleanup_vram()
            try:
                extracted = run_extraction(session_id, turn_count, full_response, player_input)
                print(f"[extraction] {len(extracted.get('events',[]))} eventos, {len(extracted.get('entities',[]))} entidades")

                l1_insert_turn(
                    session_id=session_id,
                    turn=turn_count,
                    player_input=player_input,
                    dm_response=full_response,
                    arbiter_ruling=arbiter_ruling,
                    npc_responses=npc_response,
                    world_state=world_state,
                )
            except Exception as exc:
                print(f"[post-turno] Error: {exc}")
            finally:
                _cleanup_vram()

        _last_post_turn = Thread(target=_post_turn_work, daemon=True)
        _last_post_turn.start()

    except Exception as exc:
        tb = traceback.format_exc()
        print(f"Error pipeline MAS:\n{tb}")
        _cleanup_vram()
        q.put({"type": "error", "error": str(exc)})
        q.put(None)


# ═══════════════════════════════════════════════════════════════════════
# Endpoints
# ═══════════════════════════════════════════════════════════════════════

@app.get("/health")
async def health():
    free, total = torch.cuda.mem_get_info(0)
    return {
        "status": "ok",
        "mode": "mas-multiagente",
        "model": BASE_MODEL_ID,
        "gpu": torch.cuda.get_device_name(0),
        "vram_free_gb": round(free / 1024**3, 1),
        "vram_total_gb": round(total / 1024**3, 1),
        "agents": ["router", "reglas", "npc", "mundo", "narrador", "extracción"],
        "memory_layers": ["L1-working", "L2-episodic", "L3-semantic", "L4-procedural"],
    }


STREAM_TIMEOUT_S = 120


@app.post("/generate")
async def generate(body: dict[str, Any]) -> StreamingResponse:
    session_id = body.get("sessionId", body.get("session_id", "?"))
    print(f"\n{'='*50}")
    print(f"[MAS] Request | session={session_id}")

    try:
        snake_body = _convert_keys(body)
        request = DmModelRequest(**snake_body)
    except Exception as exc:
        error = json.dumps({"type": "error", "error": f"Input inválido: {exc}"})
        return StreamingResponse(iter([f"data: {error}\n\n"]), media_type="text/event-stream")

    chunk_queue: queue.Queue = queue.Queue()
    loop = asyncio.get_event_loop()
    loop.run_in_executor(_executor, _run_mas_pipeline, request, chunk_queue)

    async def stream() -> AsyncGenerator[str, None]:
        deadline = time.monotonic() + STREAM_TIMEOUT_S
        while True:
            if time.monotonic() > deadline:
                yield f"data: {json.dumps({'type': 'error', 'error': 'Timeout: el modelo tardó demasiado en responder'})}\n\n"
                break
            try:
                item = chunk_queue.get_nowait()
            except queue.Empty:
                yield ": keepalive\n\n"
                await asyncio.sleep(2)
                continue
            if item is None:
                break
            yield f"data: {json.dumps(item)}\n\n"

    return StreamingResponse(stream(), media_type="text/event-stream")


print("Servidor FastAPI (MAS) definido — todo Gemma 4, sin APIs externas.")

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print("=" * 60)
print(f"SERVIDOR MAS PÚBLICO: {public_url}")
print("=" * 60)
print(f"\nPon esto en tu .dev.vars del backend:")
print(f"  DM_MODEL_ENDPOINT_MAS={public_url}")
print(f"  DM_USE_RUNPOD=false")

In [ ]:
# Iniciar servidor (esta celda bloquea — el servidor queda corriendo)
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()